# Тема: Завантаження даних та основи SQL. DQL-команди
## Домашнє завдання № 2 (Тема 3)

**Варіант:** D - UCI Seoul Bike Sharing Demand

# **!!! NOTE: Для виконання Colab notebook треба вибрати версію середовища виконання  2026.04.**

## 1. Setup і завантаження

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 76.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import pgserver

from ucimlrepo import fetch_ucirepo
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
# fetch dataset
seoul_bike_sharing_demand = fetch_ucirepo(id=560)

bike_df = seoul_bike_sharing_demand.data.original
display(bike_df.head())

csv_path = '/content/hw3_dataset.csv'
bike_df.to_csv(csv_path, index=False)

,Date,Rented Bike Count,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day
0,1/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [4]:
bike_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   8760 non-null   object 
 1   Rented Bike Count      8760 non-null   int64  
 2   Hour                   8760 non-null   int64  
 3   Temperature            8760 non-null   float64
 4   Humidity               8760 non-null   int64  
 5   Wind speed             8760 non-null   float64
 6   Visibility             8760 non-null   int64  
 7   Dew point temperature  8760 non-null   float64
 8   Solar Radiation        8760 non-null   float64
 9   Rainfall               8760 non-null   float64
 10  Snowfall               8760 non-null   float64
 11  Seasons                8760 non-null   object 
 12  Holiday                8760 non-null   object 
 13  Functioning Day        8760 non-null   object 
dtypes: float64(6), int64(4), object(4)
memory usage: 958.3+ 

### SQL: staging-таблиця

Ми спеціально вказали TEXT для всіх колонок у SQL, це нормальна практика для staging-таблиці - проміжної таблиці для завантаження сирих даних із CSV. Після очищення перетворюємо колонки на відповідні типи (DATE, INTEGER, NUMERIC тощо) та поміщаємо в clean таблицю.

In [5]:
sql = """
DROP TABLE IF EXISTS hw3_bike_clean CASCADE;
DROP TABLE IF EXISTS hw3_bike_staging CASCADE;

CREATE TABLE hw3_bike_staging (
    "Date_raw" TEXT,
    "Rented Bike Count_raw" TEXT,
    "Hour_raw" TEXT,
    "Temperature_raw" TEXT,
    "Humidity_raw" TEXT,
    "Wind speed_raw" TEXT,
    "Visibility_raw" TEXT,
    "Dew point temperature_raw" TEXT,
    "Solar Radiation_raw" TEXT,
    "Rainfall_raw" TEXT,
    "Snowfall_raw" TEXT,
    "Seasons_raw" TEXT,
    "Holiday_raw" TEXT,
    "Functioning Day_raw" TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(sql))

print("Staging table created successfully!")

Staging table created successfully!


### COPY FROM STDIN

In [6]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_bike_staging (
                    "Date_raw",
                    "Rented Bike Count_raw",
                    "Hour_raw",
                    "Temperature_raw",
                    "Humidity_raw",
                    "Wind speed_raw",
                    "Visibility_raw",
                    "Dew point temperature_raw",
                    "Solar Radiation_raw",
                    "Rainfall_raw",
                    "Snowfall_raw",
                    "Seasons_raw",
                    "Holiday_raw",
                    "Functioning Day_raw"
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql('SELECT COUNT(*) AS loaded_rows FROM hw3_bike_staging', engine)

,loaded_rows
0,8760


## 2. Staging і DDL

Датасет містить 8760 рядків (365 днів × 24 години). Кожен рядок описує одну годину оренди велосипедів разом із погодою в Сеулі. Усі колонки staging мають тип TEXT, бо при імпорті ми нічого не перетворюємо й не губимо.

### Пояснення до таблиць

Одиниці виміру взято з опису датасету на UCI. Вони винесені в назви колонок (`_c`, `_pct`, `_ms`).

#### Час

| Колонка | Тип | Початковий формат |
|---|---|---|
| `rent_date` | `DATE` | Текст `DD/MM/YYYY` |
| `rent_hour` | `SMALLINT` | Ціле як текст, 0-23 |
| `observed_at` | `TIMESTAMPTZ` | Похідна: дата + година |

- **`rent_date`** - календарний день спостереження. `to_date(TRIM(...), 'DD/MM/YYYY')`: формат вказано явно, бо `01/12` без маски можна прочитати як 12 січня. Час не потрібен, тож `DATE`, а не `TIMESTAMP`. `NOT NULL`.
- **`rent_hour`** - година доби, до якої належить вимір. `NULLIF(TRIM)` + каст, `CHECK (0..23)`, `NOT NULL`. `SMALLINT` вистачає для 24 значень.
- **`observed_at`** - момент початку години. Будується як `(date + hour) AT TIME ZONE 'Asia/Seoul'`: сирі дані містять локальний час Сеула без зсуву, тож локалізуємо явно. У Південній Кореї немає літнього часу, тому години не дублюються і не пропадають. `UNIQUE` гарантує одну годину в одному записі.

#### Цільова змінна

| Колонка | Тип | Початковий формат |
|---|---|---|
| `rented_bike_count` | `INTEGER` | Ціле як текст |

- **`rented_bike_count`** - кількість орендованих велосипедів за годину. Каст, `CHECK (>= 0)`, `NOT NULL` (без цієї величини рядок втрачає зміст). Це не гроші й не дробове значення, тому `INTEGER`.

#### Числові погодні показники

| Колонка | Одиниця | Тип |
|---|---|---|
| `temperature_c` | °C | `NUMERIC(4,1)` |
| `dew_point_c` | °C | `NUMERIC(4,1)` |
| `humidity_pct` | % | `SMALLINT` |
| `wind_speed_ms` | м/с | `NUMERIC(4,1)` |
| `visibility_10m` | 10 м | `INTEGER` |
| `solar_radiation_mj_m2` | МДж/м² | `NUMERIC(4,2)` |
| `rainfall_mm` | мм | `NUMERIC(5,1)` |
| `snowfall_cm` | см | `NUMERIC(4,1)` |

- **`temperature_c`, `dew_point_c`** - температура повітря і точка роси. `NULLIF(TRIM)` + каст. Одне десяткове число, тож `NUMERIC` без похибок `FLOAT`. `CHECK (-60..60)` відсікає сенсорні збої.
- **`humidity_pct`** - відносна вологість. Ціле число, `CHECK (0..100)`.
- **`wind_speed_ms`** - швидкість вітру, `CHECK (>= 0)`.
- **`visibility_10m`** - видимість. Одиницю збережено в назві, а не конвертовано в метри, щоб не змінювати дані без потреби. `CHECK (>= 0)`.
- **`solar_radiation_mj_m2`** - єдиний показник із двома десятковими знаками. `CHECK (>= 0)`. Уночі 0 є нормою, а не пропуском, тому на `NULL` не замінюється.
- **`rainfall_mm`, `snowfall_cm`** - опади за годину. `CHECK (>= 0)`. 0 означає "опадів не було", а не пропуск.

Усі ці колонки nullable: пропущений погодний вимір не робить рядок безглуздим. Порожній рядок `''` перетворюється на `NULL` через `NULLIF`, інакше каст упав би.

#### Категоріальні та бінарні ознаки

| Колонка | Тип | Початковий формат |
|---|---|---|
| `season` | `TEXT` + `CHECK` | `Winter` / `Spring` / `Summer` / `Autumn` |
| `is_holiday` | `BOOLEAN` | `Holiday` / `No Holiday` |
| `is_functioning_day` | `BOOLEAN` | `Yes` / `No` |

- **`season`** - пора року. `INITCAP(TRIM(...))` нормалізує регістр. Набір значень закритий і малий, тому `TEXT` з `CHECK IN (...)` простіший за `ENUM` і не потребує `ALTER TYPE`. `NOT NULL`. `VARCHAR(n)` не потрібен: бізнес-ліміту довжини немає.
- **`is_holiday`** - чи є день державним святом. `CASE LOWER(TRIM(...))`: `holiday` → `TRUE`, `no holiday` → `FALSE`. Інше значення дає `NULL` і відсікається через `NOT NULL`, тому помилка в даних не замаскується під `FALSE`.
- **`is_functioning_day`** - чи працювала система прокату. Та сама логіка: `yes` → `TRUE`, `no` → `FALSE`. Правило предметної області: `CHECK (is_functioning_day OR rented_bike_count = 0)`.

#### Технічний ключ

- **`record_id`** (`BIGINT GENERATED BY DEFAULT AS IDENTITY`) - суррогатний первинний ключ. Природний ключ `(rent_date, rent_hour)` теж унікальний, це закріплено через `UNIQUE`, але технічний ключ компактніший для зв'язків.

#### Підсумок щодо вимог

- **Staging і final:** `hw3_bike_staging` (усі колонки `TEXT`) та `hw3_bike_clean` (типізована таблиця).
- **Ключ:** `record_id` (identity `PRIMARY KEY`) плюс `UNIQUE (rent_date, rent_hour)`.
- **`NOT NULL`:** дата, година, момент часу, кількість оренд, сезон і обидва прапорці.
- **`CHECK`:** 11 обмежень на рівні стовпців (діапазон годин, невід'ємність лічильників і опадів, діапазони вологості та температури, закритий набір сезонів) плюс правило предметної області про неробочий день.
- **`NUMERIC(p,s)`:** для точних десяткових вимірювань. Грошових колонок у датасеті немає, тому для погоди використано `NUMERIC`, а не `FLOAT`.
- **`DATE`:** `rent_date`.
- **`TIMESTAMPTZ`:** `observed_at`.
- **`BOOLEAN`:** `is_holiday`, `is_functioning_day`.
- **`TEXT`:** `season` (із `CHECK IN (...)`).

### Створення clean таблиці

In [7]:
CLEAN_DDL = """
DROP TABLE IF EXISTS hw3_bike_clean CASCADE;

CREATE TABLE hw3_bike_clean (
    record_id             BIGINT
        GENERATED BY DEFAULT AS IDENTITY
        PRIMARY KEY,

    rent_date             DATE          NOT NULL,
    rent_hour             SMALLINT      NOT NULL
        CHECK (rent_hour BETWEEN 0 AND 23),
    observed_at           TIMESTAMPTZ   NOT NULL,

    rented_bike_count     INTEGER       NOT NULL
        CHECK (rented_bike_count >= 0),

    temperature_c         NUMERIC(4, 1)
        CHECK (temperature_c BETWEEN -60 AND 60),
    humidity_pct          SMALLINT
        CHECK (humidity_pct BETWEEN 0 AND 100),
    wind_speed_ms         NUMERIC(4, 1)
        CHECK (wind_speed_ms >= 0),
    visibility_10m        INTEGER
        CHECK (visibility_10m >= 0),
    dew_point_c           NUMERIC(4, 1)
        CHECK (dew_point_c BETWEEN -60 AND 60),
    solar_radiation_mj_m2 NUMERIC(4, 2)
        CHECK (solar_radiation_mj_m2 >= 0),
    rainfall_mm           NUMERIC(5, 1)
        CHECK (rainfall_mm >= 0),
    snowfall_cm           NUMERIC(4, 1)
        CHECK (snowfall_cm >= 0),

    season                TEXT          NOT NULL
        CHECK (season IN ('Winter', 'Spring', 'Summer', 'Autumn')),
    is_holiday            BOOLEAN       NOT NULL,
    is_functioning_day    BOOLEAN       NOT NULL,

    CONSTRAINT uq_hw3_clean_date_hour UNIQUE (rent_date, rent_hour),
    CONSTRAINT uq_hw3_clean_observed_at UNIQUE (observed_at),
    CONSTRAINT chk_hw3_no_rentals_if_not_functioning
        CHECK (is_functioning_day OR rented_bike_count = 0)
);
"""

with engine.begin() as conn:
    conn.execute(text(CLEAN_DDL))

### staging → clean

In [8]:
STAGING_TO_CLEAN_SQL = """
WITH parsed AS (
    SELECT
        to_date(NULLIF(TRIM("Date_raw"), ''), 'DD/MM/YYYY')   AS rent_date,
        NULLIF(TRIM("Hour_raw"), '')::SMALLINT                AS rent_hour,
        NULLIF(TRIM("Rented Bike Count_raw"), '')::INTEGER    AS rented_bike_count,
        NULLIF(TRIM("Temperature_raw"), '')::NUMERIC(4, 1)    AS temperature_c,
        NULLIF(TRIM("Humidity_raw"), '')::SMALLINT            AS humidity_pct,
        NULLIF(TRIM("Wind speed_raw"), '')::NUMERIC(4, 1)     AS wind_speed_ms,
        NULLIF(TRIM("Visibility_raw"), '')::INTEGER           AS visibility_10m,
        NULLIF(TRIM("Dew point temperature_raw"), '')::NUMERIC(4, 1) AS dew_point_c,
        NULLIF(TRIM("Solar Radiation_raw"), '')::NUMERIC(4, 2) AS solar_radiation_mj_m2,
        NULLIF(TRIM("Rainfall_raw"), '')::NUMERIC(5, 1)       AS rainfall_mm,
        NULLIF(TRIM("Snowfall_raw"), '')::NUMERIC(4, 1)       AS snowfall_cm,
        INITCAP(NULLIF(TRIM("Seasons_raw"), ''))              AS season,
        CASE LOWER(TRIM("Holiday_raw"))
            WHEN 'holiday'    THEN TRUE
            WHEN 'no holiday' THEN FALSE
        END AS is_holiday,
        CASE LOWER(TRIM("Functioning Day_raw"))
            WHEN 'yes' THEN TRUE
            WHEN 'no'  THEN FALSE
        END AS is_functioning_day
    FROM hw3_bike_staging
)
INSERT INTO hw3_bike_clean (
    rent_date, rent_hour, observed_at, rented_bike_count,
    temperature_c, humidity_pct, wind_speed_ms, visibility_10m,
    dew_point_c, solar_radiation_mj_m2, rainfall_mm, snowfall_cm,
    season, is_holiday, is_functioning_day
)
SELECT
    rent_date,
    rent_hour,
    (rent_date + make_interval(hours => rent_hour::INT)) AT TIME ZONE 'Asia/Seoul',
    rented_bike_count,
    temperature_c, humidity_pct, wind_speed_ms, visibility_10m,
    dew_point_c, solar_radiation_mj_m2, rainfall_mm, snowfall_cm,
    season, is_holiday, is_functioning_day
FROM parsed
WHERE rent_date IS NOT NULL
  AND rent_hour BETWEEN 0 AND 23
  AND rented_bike_count >= 0
  AND season IS NOT NULL
  AND is_holiday IS NOT NULL
  AND is_functioning_day IS NOT NULL
  AND (is_functioning_day OR rented_bike_count = 0);
"""

with engine.begin() as conn:
    result = conn.execute(text(STAGING_TO_CLEAN_SQL))
    print("Number of rows inserted:", result.rowcount)

Number of rows inserted: 8760


### Перевірка створених даних у clean таблиці

In [9]:
with engine.connect() as conn:
    counts = conn.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM hw3_bike_staging) AS staging_rows,
            (SELECT COUNT(*) FROM hw3_bike_clean)   AS clean_rows
    """)).one()
    print(counts)

pd.read_sql("SELECT * FROM hw3_bike_clean ORDER BY observed_at LIMIT 10", engine)

(8760, 8760)


,record_id,rent_date,rent_hour,observed_at,rented_bike_count,temperature_c,humidity_pct,wind_speed_ms,visibility_10m,dew_point_c,solar_radiation_mj_m2,rainfall_mm,snowfall_cm,season,is_holiday,is_functioning_day
0,1,2017-12-01,0,2017-11-30 15:00:00+00:00,254,-5.2,37,2.2,2000,-17.6,0.00,0.0,0.0,Winter,False,True
1,2,2017-12-01,1,2017-11-30 16:00:00+00:00,204,-5.5,38,0.8,2000,-17.6,0.00,0.0,0.0,Winter,False,True
2,3,2017-12-01,2,2017-11-30 17:00:00+00:00,173,-6.0,39,1.0,2000,-17.7,0.00,0.0,0.0,Winter,False,True
3,4,2017-12-01,3,2017-11-30 18:00:00+00:00,107,-6.2,40,0.9,2000,-17.6,0.00,0.0,0.0,Winter,False,True
4,5,2017-12-01,4,2017-11-30 19:00:00+00:00,78,-6.0,36,2.3,2000,-18.6,0.00,0.0,0.0,Winter,False,True
5,6,2017-12-01,5,2017-11-30 20:00:00+00:00,100,-6.4,37,1.5,2000,-18.7,0.00,0.0,0.0,Winter,False,True
6,7,2017-12-01,6,2017-11-30 21:00:00+00:00,181,-6.6,35,1.3,2000,-19.5,0.00,0.0,0.0,Winter,False,True
7,8,2017-12-01,7,2017-11-30 22:00:00+00:00,460,-7.4,38,0.9,2000,-19.3,0.00,0.0,0.0,Winter,False,True
8,9,2017-12-01,8,2017-11-30 23:00:00+00:00,930,-7.6,37,1.1,2000,-19.8,0.01,0.0,0.0,Winter,False,True
9,10,2017-12-01,9,2017-12-01 00:00:00+00:00,490,-6.5,27,0.5,1928,-22.4,0.23,0.0,0.0,Winter,False,True


## 3. Data-quality audit

### 3.1 Контроль кількості рядків

In [10]:
pd.read_sql("""
    SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_bike_staging
    UNION ALL
    SELECT 'clean'   AS layer, COUNT(*) AS n_rows FROM hw3_bike_clean
""", engine)

,layer,n_rows
0,staging,8760
1,clean,8760


| Шар | Кількість рядків |
|---|---|
| staging | 8760 |
| clean | 8760 |

Кількість рядків збігається: жоден запис не відфільтровано. Усі сім лічильників причин відсіву дорівнюють 0. Правила фільтрації в `INSERT ... SELECT`: рядок відкидається, якщо немає дати, години (або вона поза 0-23), кількості оренд, сезону чи одного з булевих прапорців, а також якщо у неробочий день кількість оренд більша за 0. Збіг 8760 = 365 × 24 відповідає повному року погодинних даних.

### 3.2 Перевірка дублікатів

In [11]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (rent_date, rent_hour)) AS distinct_business_keys,
        COUNT(*) - COUNT(DISTINCT (rent_date, rent_hour)) AS duplicate_rows
    FROM hw3_bike_clean
""", engine)

,total_rows,distinct_business_keys,duplicate_rows
0,8760,8760,0


Business key: `(rent_date, rent_hour)`: одна година однієї доби.

| total_rows | distinct_business_keys | duplicate_rows |
|---|---|---|
| 8760 | 8760 | 0 |

Дублікатів немає. Це не випадковість: у таблиці вже діє `UNIQUE (rent_date, rent_hour)`, тож дублікат зупинив би `INSERT`. Перевірка показує, що обмеження не довелося "обходити". Кількість рядків дорівнює очікуваній кількості годин у діапазоні дат, отже пропущених годин також немає.

Обмеження: ключ `(дата, година)` є справжнім природним ключем лише тому, що датасет має один рядок на годину для всього міста. Якби дані збиралися по кількох станціях, цей ключ перестав би бути унікальним.

### 3.3. Null-rate і спеціальні маркери

In [12]:
null_rates = pd.read_sql("""
    SELECT
        ROUND(COUNT(*) FILTER (WHERE rented_bike_count IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_bike_count,
        ROUND(COUNT(*) FILTER (WHERE temperature_c IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_temperature,
        ROUND(COUNT(*) FILTER (WHERE humidity_pct IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_humidity,
        ROUND(COUNT(*) FILTER (WHERE wind_speed_ms IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_wind,
        ROUND(COUNT(*) FILTER (WHERE visibility_10m IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_visibility,
        ROUND(COUNT(*) FILTER (WHERE solar_radiation_mj_m2 IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_solar,
        ROUND(COUNT(*) FILTER (WHERE rainfall_mm IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_rainfall,
        ROUND(COUNT(*) FILTER (WHERE snowfall_cm IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_snowfall,
        COUNT(*) AS total_rows
    FROM hw3_bike_clean
""", engine)
null_rates.T # транспонуємо, щоб влізло в екран

,0
pct_null_bike_count,0.0
pct_null_temperature,0.0
pct_null_humidity,0.0
pct_null_wind,0.0
pct_null_visibility,0.0
pct_null_solar,0.0
pct_null_rainfall,0.0
pct_null_snowfall,0.0
total_rows,8760.0


Спеціальні маркери у staging (сирі дані). Ми перевіряємо порожній рядок, пробіли та типові маркери пропуску (?, unknown, na, n/a, nan, null, none, -) у кожній колонці.

In [13]:
pd.read_sql("""
WITH long AS (
    SELECT 'Date'              AS col, "Date_raw"                  AS v FROM hw3_bike_staging UNION ALL
    SELECT 'Rented Bike Count',        "Rented Bike Count_raw"          FROM hw3_bike_staging UNION ALL
    SELECT 'Hour',                     "Hour_raw"                       FROM hw3_bike_staging UNION ALL
    SELECT 'Temperature',              "Temperature_raw"                FROM hw3_bike_staging UNION ALL
    SELECT 'Humidity',                 "Humidity_raw"                   FROM hw3_bike_staging UNION ALL
    SELECT 'Wind speed',               "Wind speed_raw"                 FROM hw3_bike_staging UNION ALL
    SELECT 'Visibility',               "Visibility_raw"                 FROM hw3_bike_staging UNION ALL
    SELECT 'Dew point temperature',    "Dew point temperature_raw"      FROM hw3_bike_staging UNION ALL
    SELECT 'Solar Radiation',          "Solar Radiation_raw"            FROM hw3_bike_staging UNION ALL
    SELECT 'Rainfall',                 "Rainfall_raw"                   FROM hw3_bike_staging UNION ALL
    SELECT 'Snowfall',                 "Snowfall_raw"                   FROM hw3_bike_staging UNION ALL
    SELECT 'Seasons',                  "Seasons_raw"                    FROM hw3_bike_staging UNION ALL
    SELECT 'Holiday',                  "Holiday_raw"                    FROM hw3_bike_staging UNION ALL
    SELECT 'Functioning Day',          "Functioning Day_raw"            FROM hw3_bike_staging
)
SELECT
    col,
    COUNT(*) FILTER (WHERE v IS NULL)                                   AS sql_null,
    COUNT(*) FILTER (WHERE TRIM(v) = '')                                AS empty_or_blank,
    COUNT(*) FILTER (WHERE LOWER(TRIM(v)) IN
        ('?', 'unknown', 'na', 'n/a', 'nan', 'null', 'none', '-', '--')) AS marker_values
FROM long
GROUP BY col
ORDER BY col;
""", engine)

,col,sql_null,empty_or_blank,marker_values
0,Date,0,0,0
1,Dew point temperature,0,0,0
2,Functioning Day,0,0,0
3,Holiday,0,0,0
4,Hour,0,0,0
5,Humidity,0,0,0
6,Rainfall,0,0,0
7,Rented Bike Count,0,0,0
8,Seasons,0,0,0
9,Snowfall,0,0,0


**Null-rate у `hw3_bike_clean`** (перевірено 8 колонок):

| Колонка | % NULL |
|---|---|
| `rented_bike_count` | 0.00 |
| `temperature_c` | 0.00 |
| `humidity_pct` | 0.00 |
| `wind_speed_ms` | 0.00 |
| `visibility_10m` | 0.00 |
| `solar_radiation_mj_m2` | 0.00 |
| `rainfall_mm` | 0.00 |
| `snowfall_cm` | 0.00 |

Нульовий null-rate є результатом, а не відсутністю перевірки. Він збігається з `bike_df.info()`: у всіх 14 колонках 8760 непорожніх значень.

**Сирі дані (staging):** порожніх рядків, пробілів і маркерів `?`, `unknown`, `NA` та подібних не знайдено ні в одній колонці. Штучних `NULL` ми не створювали.

**Чому це не гарантує повної якості.** Пропуски в цьому датасеті можуть бути замасковані реальними значеннями:

- `rented_bike_count = 0` трапляється у неробочі дні. Це наслідок того, що система не працювала, а не "нуль оренд за робочої системи". Для моделювання попиту такі рядки варто виключати або розглядати окремо.
- Нулі в `solar_radiation_mj_m2` вночі, а також у `rainfall_mm` і `snowfall_cm` без опадів є фізично коректними, тому їх не можна вважати пропусками.
- Максимальне значення `visibility_10m` (2000) може бути стелею вимірювального приладу, а не реальною видимістю.
- Відсутність `NULL` нічого не каже про точність вимірювань, про викиди в межах дозволених діапазонів і про те, чи правильно датчик калібровано.

## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки

In [14]:
display(pd.read_sql("""
    SELECT season, COUNT(*) AS n_rows,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
    FROM hw3_bike_clean
    GROUP BY season
    ORDER BY n_rows DESC, season
""", engine))

display(pd.read_sql("""
    SELECT is_holiday, COUNT(*) AS n_rows,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows,
           COUNT(DISTINCT rent_date) AS n_days
    FROM hw3_bike_clean
    GROUP BY is_holiday
    ORDER BY n_rows DESC
""", engine))

,season,n_rows,pct_rows
0,Spring,2208,25.21
1,Summer,2208,25.21
2,Autumn,2184,24.93
3,Winter,2160,24.66


,is_holiday,n_rows,pct_rows,n_days
0,False,8328,95.07,347
1,True,432,4.93,18


**`season`** (очікуваний розподіл):

| season | n_rows | pct_rows |
|---|---|---|
| Spring | 2208 | 25.21 |
| Summer | 2208 | 25.21 |
| Autumn | 2184 | 24.93 |
| Winter | 2160 | 24.66 |

Сезони майже рівномірні: кожен займає близько чверті року. Різниця в кількості рядків пояснюється лише різною тривалістю сезонів (90-92 доби × 24 години), а не пропусками. Рідкісних категорій немає, написання значень однакове, тож нормалізація `INITCAP` спрацювала коректно.

**`is_holiday`** (очікувано): `FALSE` становить 95.07% рядків, `TRUE` лише 4.93% (18 святкових днів × 24 години = 432 рядків). Це природний дисбаланс: свят у році мало. Для аналізу попиту святкові дні варто порівнювати з будніми обережно, бо вибірка невелика, і висновки про "поведінку у свята" спираються лише на кілька десятків спостережень на кожну пору дня.

### 4.2. Базова статистика числової колонки

Колонка: `rented_bike_count`, цільова змінна всього датасету. Медіана тут не обов'язкова, але вона допомагає побачити асиметрію: якщо середнє помітно більше за медіану, розподіл скошений вправо.

In [15]:
pd.read_sql("""
    SELECT
        COUNT(rented_bike_count) AS n_non_null,
        MIN(rented_bike_count)   AS min_value,
        MAX(rented_bike_count)   AS max_value,
        ROUND(AVG(rented_bike_count)::NUMERIC, 2)         AS avg_value,
        ROUND(STDDEV_SAMP(rented_bike_count)::NUMERIC, 2) AS sd_value,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rented_bike_count) AS median_value
    FROM hw3_bike_clean
""", engine).T

,0
n_non_null,8760.0
min_value,0.0
max_value,3556.0
avg_value,704.6
sd_value,645.0
median_value,504.5


| Показник | Значення (очікувано) |
|---|---|
| n_non_null | 8760 |
| min | 0 |
| max | 3556 |
| avg | ≈ 704.6 |
| sd | ≈ 645 |
| median | 504.5 |

Мінімум 0 відповідає неробочим годинам і ночі. Стандартне відхилення майже дорівнює середньому, а середнє помітно більше за медіану, тобто розподіл скошений вправо: більшість годин мають помірний попит, а невелика кількість годин (вечірній пік у теплу пору) мають дуже високий. Для такої змінної середнє й σ описують дані гірше, ніж медіана й квантилі, тому правило 3σ тут потрібно застосовувати з обережністю.

### 4.3 Outlier-check: доменний варіант

#### Статистична перевірка 3σ

Рядки з цього запиту, припадають на теплі вечірні години робочих днів. Рекомендована дія: залишити як валідний рідкісний випадок. Це справжній пік попиту (час після роботи), а не помилка вимірювання, і модель має його бачити.

In [16]:
three_s_sql = """
WITH stats AS (
    SELECT AVG(rented_bike_count) AS mean_value,
           STDDEV_SAMP(rented_bike_count) AS sd_value
    FROM hw3_bike_clean
    WHERE rented_bike_count IS NOT NULL
)
SELECT
    t.rent_date, t.rent_hour, t.rented_bike_count, t.season, t.temperature_c, t.is_holiday,
    ROUND(((t.rented_bike_count - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score
FROM hw3_bike_clean AS t
CROSS JOIN stats AS s
WHERE s.sd_value > 0
  AND ABS(t.rented_bike_count - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.rented_bike_count - s.mean_value) DESC
LIMIT 20;
"""
pd.read_sql(three_s_sql, engine)

,rent_date,rent_hour,rented_bike_count,season,temperature_c,is_holiday,z_score
0,2018-06-19,18,3556,Summer,24.1,False,4.42
1,2018-06-21,18,3418,Summer,27.8,False,4.21
2,2018-06-12,18,3404,Summer,24.9,False,4.19
3,2018-06-20,18,3384,Summer,27.0,False,4.15
4,2018-06-04,18,3380,Summer,24.4,False,4.15
5,2018-06-22,18,3365,Summer,29.3,False,4.12
6,2018-06-08,18,3309,Summer,26.2,False,4.04
7,2018-09-10,18,3298,Autumn,25.9,False,4.02
8,2018-09-17,18,3277,Autumn,25.3,False,3.99
9,2018-09-12,18,3256,Autumn,27.0,False,3.96


#### Доменна перевірка

Обираю доменну перевірку, а не 3σ. Причина: `rented_bike_count` сильно скошена вправо, і правило 3σ позначило б як "викиди" цілком нормальні вечірні піки літніх днів. Доменні правила перевіряють те, що фізично чи логічно неможливо, незалежно від розподілу.

Перевіряємо чотири правила:

1. Точка роси не може перевищувати температуру повітря.
2. Сонячна радіація вночі (години 0-3) має дорівнювати нулю.
3. Одночасні дощ і сніг в одну годину: рідкісно, але можливо (мокрий сніг), тож це кандидат на перевірку, а не помилка.
4. Сезон має відповідати місяцю (метеорологічні сезони: грудень-лютий зима, березень-травень весна, червень-серпень літо, вересень-листопад осінь).

In [17]:
audit_sql = """
WITH checks AS (
    SELECT
        (dew_point_c > temperature_c)                                 AS dew_above_temp,
        (rent_hour BETWEEN 0 AND 3 AND solar_radiation_mj_m2 > 0)     AS night_sun,
        (rainfall_mm > 0 AND snowfall_cm > 0)                         AS rain_and_snow,
        (season <> CASE
            WHEN EXTRACT(MONTH FROM rent_date) IN (12, 1, 2)  THEN 'Winter'
            WHEN EXTRACT(MONTH FROM rent_date) IN (3, 4, 5)   THEN 'Spring'
            WHEN EXTRACT(MONTH FROM rent_date) IN (6, 7, 8)   THEN 'Summer'
            ELSE 'Autumn'
        END)                                                          AS season_month_mismatch
    FROM hw3_bike_clean
)
SELECT
    COUNT(*) FILTER (WHERE dew_above_temp)        AS dew_above_temp,
    COUNT(*) FILTER (WHERE night_sun)             AS night_sun,
    COUNT(*) FILTER (WHERE rain_and_snow)         AS rain_and_snow,
    COUNT(*) FILTER (WHERE season_month_mismatch) AS season_month_mismatch,
    COUNT(*)                                      AS total_rows
FROM checks
"""
pd.read_sql(audit_sql, engine).T

,0
dew_above_temp,0
night_sun,0
rain_and_snow,28
season_month_mismatch,0
total_rows,8760


Виконано **доменний** outlier-check із чотирьох правил:

| Правило | Знайдено рядків |
|---|---|
| Точка роси вища за температуру повітря | 0 |
| Сонячна радіація > 0 о 00:00-03:59 | 0 |
| Дощ і сніг одночасно в одну годину | 28 |
| Сезон не відповідає місяцю | 0 |


Майже всі лічильники (`dew_above_temp`, `night_sun`, `season_month_mismatch`) дорівнюють 0, тобто перевірка не знайшла фізично чи логічно неможливих комбінацій. Це означає лише те, що ці три правила виконуються.

**Є ненульове значення (`28`) для лічильника "Дощ і сніг одночасно в одну годину". Пропозиція - залишити як валідний рідкісний випадок (мокрий сніг), за потреби додати прапорець.**

Великі значення `rented_bike_count` за цим аудитом аномаліями не вважаються: це реальні пікові години, і їх видаляти не можна.

## 5. Десять DQL/EDA-запитів

### 5.1 Години з найбільшим попитом

**Business-question:** У які п'ять годин доби в середньому орендують найбільше велосипедів?

In [18]:
top_5_rent_hours_sql = """
SELECT
    rent_hour,
    ROUND(AVG(rented_bike_count), 1) AS avg_bikes,
    SUM(rented_bike_count)           AS total_bikes
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY rent_hour
ORDER BY avg_bikes DESC, rent_hour
LIMIT 5;
"""
pd.read_sql(top_5_rent_hours_sql, engine)

,rent_hour,avg_bikes,total_bikes
0,18,1554.0,548568
1,19,1235.8,436229
2,17,1177.2,415556
3,20,1105.3,390172
4,21,1066.5,376479


**Interpretation:** Пʼять найпопулярніших годин для оренди велосипеда (в дужках біля години вказано середня кількість велосипедів орендована в цій годині): 18 (1554), 19 (1235.8), 17 (1177.2), 20 (1105.3), 21 (1066.5). Як бачимо пік припадає на вечірній час після роботи. О 18 годині оренда велосипедів на приблизно 31% популярніша, ніж о 21 годині (пʼятій за популярністю). Неробочі дні виключено через WHERE is_functioning_day, інакше нулі б занизили середні значення.

### 5.2 Попит за сезонами

**Business-question:** Як змінюється середній попит на велосипеди залежно від сезону?

In [19]:
demand_by_season_sql = """
SELECT
    season,
    COUNT(DISTINCT rent_date)            AS n_days,
    SUM(rented_bike_count)               AS total_bikes,
    ROUND(AVG(rented_bike_count), 1)     AS avg_bikes_per_hour,
    ROUND(AVG(temperature_c), 1)         AS avg_temp_c
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY season
ORDER BY avg_bikes_per_hour DESC;
"""
pd.read_sql(demand_by_season_sql, engine)

,season,n_days,total_bikes,avg_bikes_per_hour,avg_temp_c
0,Summer,92,2283234,1034.1,26.6
1,Autumn,81,1790002,924.1,13.8
2,Spring,90,1611909,746.3,13.0
3,Winter,90,487169,225.5,-2.5


**Interpretation:** Найбільший попит - влітку, найменший - взимку. Різниця між ними - майже у 4.7 раз. Згідно нашої таблички - чим вища середня температура сезону, тим вища популярність оренди велосипедів. Це видно не тільки по загальній кількості орендованих велосипедах (що не зовсім правильно порівнювати по цьому показнику, оскільки сезони мають різну кількість днів), але й по середній кількості орендованих велосипедів за годину.

### 5.3 Свята проти звичайних днів

**Business-question:** Чим відрізняється попит у святкові та несвяткові дні, коли система працює?

In [20]:
working_vs_holiday_sql = """
SELECT
    is_holiday,
    COUNT(DISTINCT rent_date)                                         AS n_days,
    ROUND(AVG(rented_bike_count), 1)                                  AS avg_bikes_per_hour,
    ROUND(SUM(rented_bike_count)::NUMERIC / COUNT(DISTINCT rent_date), 0) AS avg_bikes_per_day
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY is_holiday
ORDER BY is_holiday;
"""
pd.read_sql(working_vs_holiday_sql, engine)

,is_holiday,n_days,avg_bikes_per_hour,avg_bikes_per_day
0,False,336,739.3,17727.0
1,True,17,529.2,12700.0


Додатковий запит для визначення розподілу святкових днів у які система працює по сезонам

In [21]:
holidays_by_season_sql = """
SELECT
    season,
    COUNT(*) FILTER (WHERE is_holiday) / 24 AS n_holidays,
    COUNT(DISTINCT rent_date)                                         AS n_days
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY season
ORDER BY n_holidays DESC;
"""
pd.read_sql(holidays_by_season_sql, engine)

,season,n_holidays,n_days
0,Winter,8,90
1,Autumn,4,81
2,Spring,3,90
3,Summer,2,92


**Interpretation:** Приблизно у 1.4 рази попит у свята відрізняється від звичайних днів (за годиною і за добою із майже одинаковою пропорцією). Враховуючи, що n_days святкових днів мало (17), тож висновок спирається на невелику вибірку. Різницю можна частково пояснити тим, що сезон свят переважно припадає на проходін пори року, як видно у додатковому запиті.

### 5.4 Нульові оренди у робочий день

**Business-question:** Чи бувають години з нульовими орендами тоді, коли система прокату працює?

In [22]:
zero_rentals_by_functioning_status_sql = """
SELECT
    is_functioning_day,
    COUNT(*)                                          AS n_hours,
    COUNT(*) FILTER (WHERE rented_bike_count = 0)     AS zero_hours,
    ROUND(COUNT(*) FILTER (WHERE rented_bike_count = 0) * 100.0
          / NULLIF(COUNT(*), 0), 2)                   AS pct_zero
FROM hw3_bike_clean
GROUP BY is_functioning_day
ORDER BY is_functioning_day;
"""

zero_rentals_on_functioning_days_sql = """
SELECT rent_date, rent_hour, season, rented_bike_count
FROM hw3_bike_clean
WHERE is_functioning_day AND rented_bike_count = 0
ORDER BY rent_date, rent_hour
LIMIT 20;
"""

display(pd.read_sql(zero_rentals_by_functioning_status_sql, engine))
display(pd.read_sql(zero_rentals_on_functioning_days_sql, engine))

,is_functioning_day,n_hours,zero_hours,pct_zero
0,False,295,295,100.0
1,True,8465,0,0.0


,rent_date,rent_hour,season,rented_bike_count


**Interpretation:** Другий запит повернув 0 рядків, це допустимий результат: у робочі дні кожна година має хоча б одну оренду, а всі нулі належать неробочим дням. Загалом на неробочі дні припадає 295 годин без оренд.

### 5.5 Ранкові та вечірні піки за сезонами

**Business-question:** Який середній попит у "години пік" (07-09 та 17-19) у звичайні робочі дні в кожному сезоні?

In [23]:
demands_rush_hours_sql = """
SELECT
    season,
    COUNT(*)                         AS n_hours,
    ROUND(AVG(rented_bike_count), 1) AS avg_bikes
FROM hw3_bike_clean
WHERE is_functioning_day
  AND NOT is_holiday
  AND (rent_hour BETWEEN 7 AND 9 OR rent_hour BETWEEN 17 AND 19)
GROUP BY season
ORDER BY avg_bikes DESC;
"""

pd.read_sql(demands_rush_hours_sql, engine)

,season,n_hours,avg_bikes
0,Summer,540,1471.5
1,Autumn,462,1354.5
2,Spring,522,1109.5
3,Winter,492,343.9


**Interpretation:** Популярність у години пік зберігають ту саму послідовність, що ми досліджували у 5.2 Попит за сезонами, що являється очікуваним результатом. Популярність оренди в години пік влітку приблизно у 4.28 разів вища за популярність взимку. Цей факт не можна пояснити тільки тим, що влітку годин більше, ніж у взимку, бо влітку годин більше всього приблизно у 1.1 раз. Як у випадку із 5.2, тут вирішальну роль зіграли тепліші темпаратури.

### 5.6 Вплив дощу навесні й восени

**Business-question:** Наскільки дощ знижує попит навесні та восени?

In [24]:
spring_and_autumn_demand_sql = """
SELECT
    (rainfall_mm > 0)                AS is_raining,
    COUNT(*)                         AS n_hours,
    ROUND(AVG(rented_bike_count), 1) AS avg_bikes
FROM hw3_bike_clean
WHERE season IN ('Spring', 'Autumn')
  AND is_functioning_day
  AND rainfall_mm IS NOT NULL
GROUP BY (rainfall_mm > 0)
ORDER BY is_raining;
"""

pd.read_sql(spring_and_autumn_demand_sql, engine)

,is_raining,n_hours,avg_bikes
0,False,3819,882.4
1,True,278,114.8


**Interpretation:** Навесні і восени було 278 годин із дощем і 3819 годин без дощу. У дощові години популярність оренди падає приблизно у 7.7 разів. Дощ частіше буває в певні години й сезони, тому різниця не є чистим ефектом дощу.

### 5.7 Грудень у сирих даних (LIKE / ILIKE)

**Business-question:** Чи всі записи за грудень у сирих даних мають позначку Winter?

In [25]:
labels_of_december_sql = """
SELECT
    COUNT(*)                                          AS n_december_rows,
    COUNT(DISTINCT "Date_raw")                        AS n_days,
    COUNT(*) FILTER (WHERE "Seasons_raw" ILIKE 'winter') AS labelled_winter
FROM hw3_bike_staging
WHERE "Date_raw" LIKE '__/12/____';
"""

pd.read_sql(labels_of_december_sql, engine)

,n_december_rows,n_days,labelled_winter
0,528,22,528


**Interpretation:** 528 рядків і 22 дні припадають на грудень. `labelled_winter` = `n_december_rows`, тобто мітка сезону узгоджена з місяцем і нормалізація INITCAP коректна. Запит працює зі staging, бо там дата ще текстова.

### 5.8 Сніг за сезонами (IS NOT NULL, COALESCE, NULLIF)

**Business-question:** У яких сезонах бував сніг, скільки годин і яка його типова кількість?

In [26]:
snow_by_season_sql = """
SELECT
    season,
    COUNT(*)                                          AS n_hours,
    COUNT(*) FILTER (WHERE COALESCE(snowfall_cm, 0) > 0) AS snowy_hours,
    ROUND(AVG(NULLIF(snowfall_cm, 0)), 2)             AS avg_snow_cm_when_snowing,
    COALESCE(MAX(snowfall_cm), 0)                     AS max_snow_cm
FROM hw3_bike_clean
WHERE snowfall_cm IS NOT NULL
GROUP BY season
ORDER BY snowy_hours DESC, season;
"""

pd.read_sql(snow_by_season_sql, engine)

,season,n_hours,snowy_hours,avg_snow_cm_when_snowing,max_snow_cm
0,Winter,2160,392,1.36,5.1
1,Autumn,2184,51,2.41,8.8
2,Spring,2208,0,NaN,0.0
3,Summer,2208,0,NaN,0.0


**Interpretation:**  Тільки взимку і восени випадав сніг. Частка годин коли падав сніг: зима (18%), осінь (2%). NULLIF(snowfall_cm, 0) виключає години без снігу з середнього, інакше AVG показував би середнє по всіх годинах. Там, де снігу не було, avg_snow_cm_when_snowing буде NaN: це не пропуск у даних, а відсутність підстави для обчислення середнього.

### 5.9 Свята, коли система не працювала (IS DISTINCT FROM, DISTINCT)

**Business-question:** Які святкові дні система прокату не працювала?

In [27]:
holidays_sql = """
SELECT DISTINCT
    rent_date,
    season
FROM hw3_bike_clean
WHERE is_holiday IS NOT DISTINCT FROM TRUE
  AND is_functioning_day IS DISTINCT FROM TRUE
ORDER BY rent_date;
"""

pd.read_sql(holidays_sql, engine)

,rent_date,season
0,2018-10-09,Autumn


**Interpretation:** Запит повернув одну дату - 2018-10-09. 9 жовтня 2018 року Південна Корея святкувала День хангилю (свято корейського письма). Згідно 5.3 святкових днів було 17 днів, лише в один святковий день система не працювала.

### 5.10 Десять найзавантаженіших годин

**Business-question:** Які десять конкретних годин мали найбільшу кількість оренд і за яких умов?

In [28]:
top_10_hours_sql = """
SELECT
    rent_date,
    rent_hour,
    season,
    is_holiday,
    temperature_c,
    rainfall_mm,
    rented_bike_count
FROM hw3_bike_clean
ORDER BY rented_bike_count DESC, rent_date, rent_hour
LIMIT 10;
"""

pd.read_sql(top_10_hours_sql, engine)

,rent_date,rent_hour,season,is_holiday,temperature_c,rainfall_mm,rented_bike_count
0,2018-06-19,18,Summer,False,24.1,0.0,3556
1,2018-06-21,18,Summer,False,27.8,0.0,3418
2,2018-06-12,18,Summer,False,24.9,0.0,3404
3,2018-06-20,18,Summer,False,27.0,0.0,3384
4,2018-06-04,18,Summer,False,24.4,0.0,3380
5,2018-06-22,18,Summer,False,29.3,0.0,3365
6,2018-06-08,18,Summer,False,26.2,0.0,3309
7,2018-09-10,18,Autumn,False,25.9,0.0,3298
8,2018-09-17,18,Autumn,False,25.3,0.0,3277
9,2018-09-12,18,Autumn,False,27.0,0.0,3256


**Interpretation:** Серед десяти найзавантаженіших годин є тільки 18 година. Це співпадає із тим, що ми досліджували у 5.1. Перших сім найзавантаженіших годин були влітку (всі у червні), інших три - восени (всі у вересні). Серед найзавантаженіших годин немає жодної у святковий день. Також варто зауважити, що ці значення великі, але це реальні піки попиту, тому їх не слід видаляти як викиди.

## 6. Reflection

**Проблеми якості.** Пропусків (`NULL`), маркерів `?` / `unknown` і дублікатів за ключем `(rent_date, rent_hour)` я не знайшов, тож практичні проблеми були іншими. Заголовки CSV містять символ `°`, тому файл потребує кодування `latin-1`/`cp1252`. Дата має формат `DD/MM/YYYY` без часового пояса. Булеві ознаки закодовані текстом (`Holiday` / `No Holiday`, `Yes` / `No`). Нулі в `rented_bike_count` у неробочі дні означають "система не працювала", а не "попиту немає". Значення `visibility_10m = 2000` схоже на стелю датчика. Нульовий null-rate не гарантує якості: помилки датчиків можуть лишатися в допустимих діапазонах.

**Features.** Я використав би годину, сезон, прапорець свята, температуру, вологість, вітер, видимість, сонячну радіацію, дощ і сніг, а також день тижня й місяць. Не брав би `record_id`. Точку роси варто виключити або перевірити, бо вона сильно корелює з температурою та вологістю. Рядки неробочих днів краще відфільтрувати, а не робити з `is_functioning_day` feature, бо він майже повністю визначає target.

**Leakage.** Фактична погода за ту саму годину стає відомою лише після її завершення. Для справжнього прогнозу потрібен прогноз погоди, а не виміряні значення. Lag-ознаки target слід будувати лише з минулих годин, а вибірку ділити за часом.

**Покращення pipeline.** Віддавати UTF-8, ISO-8601 час із зсувом, булеві значення як `true` / `false`, одиниці виміру в назвах полів, окремий прапорець для обрізаних значень і схему даних із автоматичною валідацією.

**ML-задача.** Це регресія погодинного попиту; з урахуванням часового порядку, по суті, forecasting.